In [ ]:
from google.colab import drive
import json
drive.mount('/content/drive')

# Create a folder for your results
import os
save_path = '/content/drive/MyDrive/jay_storm'
os.makedirs(save_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
from torch.optim.optimizer import Optimizer, required

class AdaSTORM(Optimizer):
    '''
    >>> Ada-STORM: Coordinate-wise Adaptive Stochastic Recursive Momentum.
    Based on STORM+ but with per-parameter adaptivity.
    '''

    def __init__(self, params, lr=required, weight_decay=0., init_accumulator=1., c=1.):
        if lr is not required and lr < 0.0:
            raise ValueError('Invalid learning rate: %1.1e' % lr)
        if weight_decay < 0.0:
            raise ValueError('Invalid weight decay value: %1.1e' % weight_decay)

        defaults = dict(lr=lr, weight_decay=weight_decay)
        super(AdaSTORM, self).__init__(params, defaults)

        self.c = c
        self.init_acc = init_accumulator

        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                state['d_t'] = torch.zeros_like(p.data)
                state['current_grad'] = torch.zeros_like(p.data)
                state['correction_grad'] = torch.zeros_like(p.data)

                # Ada-STORM: Per-parameter accumulators
                state['grad_accumulator'] = torch.full_like(p.data, self.init_acc)
                state['estimator_accumulator'] = torch.full_like(p.data, self.init_acc)

    def __setstate__(self, state):
        super(AdaSTORM, self).__setstate__(state)

    @torch.no_grad()
    def compute_estimator(self, normalized_norm=False, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            weight_decay = group['weight_decay']
            for p in group['params']:
                if p.grad is None:
                    continue

                p_grad = p.grad.data
                if weight_decay != 0:
                    p_grad = p_grad.add(p.data, alpha=weight_decay)

                state = self.state[p]

                # Coordinate-wise a_t calculation
                # a_t = c / (grad_acc)^(2/3)
                a_t = self.c / (state['grad_accumulator'].pow(2/3).add(1e-8))

                # Recursive Estimator: d_{t+1} = g_{t+1} + (1 - a_t)(d_t - g_t_old)
                # Note: d_t and correction_grad are vectors here
                state['current_grad'] = p_grad.detach()
                diff = state['d_t'].sub(state['correction_grad'])

                # d_t = g_{t+1} + (1 - a_t) * (d_t - g_t_old)
                state['d_t'] = (state['current_grad'] + (1.0 - a_t) * diff).detach()

                # Update estimator accumulator coordinate-wise for the next step
                # estimator_acc = estimator_acc + ||d_t||^2 / a_t
                state['estimator_accumulator'].add_(state['d_t'].pow(2) / a_t.add(1e-8))

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            loss = closure()

        for group in self.param_groups:
            weight_decay = group['weight_decay']
            lr = group['lr']

            for p in group['params']:
                if p.grad is None:
                    continue

                p_grad = p.grad.data
                if weight_decay != 0:
                    p_grad = p_grad.add(p.data, alpha=weight_decay)

                state = self.state[p]

                # 1. Store correction gradient for the NEXT compute_estimator call
                state['correction_grad'] = p_grad.detach()

                # 2. Update gradient accumulator coordinate-wise
                # grad_acc = grad_acc + ||g_t||^2
                state['grad_accumulator'].add_(p_grad.pow(2))

                # 3. Calculate Coordinate-wise Step Size (eta_t)
                # eta_t = 1 / (estimator_acc)^(1/3)
                eta_t = lr / (state['estimator_accumulator'].pow(1/3).add(1e-8))

                # 4. Parameter Update
                p.data.addcmul_(state['d_t'], eta_t, value=-1)

        return loss

    def lr(self):
        """Returns the average effective learning rate across all parameters."""
        etas = []
        for group in self.param_groups:
            for p in group['params']:
                state = self.state[p]
                if 'estimator_accumulator' in state:
                    # Calculate mean eta for this parameter
                    eta = group['lr'] / (state['estimator_accumulator'].pow(1/3).add(1e-8))
                    etas.append(eta.mean().item())
        return sum(etas) / len(etas) if etas else 0.0

In [ ]:
# 2. Set NeurIPS Plotting Style
plt.rcParams.update({
    "text.usetex": False, # Colab often lacks a full LaTeX install; use standard fonts
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 12,
    "font.size": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "lines.linewidth": 1.5
})

In [ ]:
class BenchmarkCNN(nn.Module):
    def __init__(self, in_channels=3):
        super(BenchmarkCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1) # Extra layer for complexity
        self.pool = nn.MaxPool2d(2, 2)
        # Spatial size: 32 -> 16 -> 8 -> 4 (if we pool 3 times)
        # But we'll pool twice for 28x28 (MNIST) and 32x32 (CIFAR/SVHN) compatibility
        self.fc1 = nn.Linear(128 * 7 * 7 if in_channels == 1 else 128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = F.relu(self.conv3(x)) # No pool on last conv to keep spatial res
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [ ]:
def get_loaders(dataset_name):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    if dataset_name == 'CIFAR10':
        trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    elif dataset_name == 'SVHN':
        trainset = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform)
    elif dataset_name == 'FashionMNIST':
        trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    else:
      raise ValueError(f"Unsupported dataset: {dataset_name}")
    return torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

In [ ]:
def train_model(dataset_name, opt_name, epochs=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    loader = get_loaders(dataset_name)
    in_channels = 1 if dataset_name == 'FashionMNIST' else 3
    model = BenchmarkCNN(in_channels).to(device)
    criterion = nn.CrossEntropyLoss()

    if opt_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    elif opt_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    elif opt_name == 'Ada-STORM':
        optimizer = AdaSTORM(model.parameters(), lr=0.1)

    # Path for this specific optimizer's JSON
    json_path = os.path.join(save_path, f"{opt_name}.json")

    history = []
    for epoch in range(epochs):
        epoch_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if opt_name == 'Ada-STORM':
                optimizer.step()
                optimizer.zero_grad()
                criterion(model(images), labels).backward()
                optimizer.compute_estimator()
                loss = criterion(model(images), labels) # Just for logging
            else:
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                optimizer.step()
            epoch_loss += loss.item()
        avg_loss = epoch_loss / len(loader)
        history.append(avg_loss)
        print(f"{dataset_name} | {opt_name} | Epoch {epoch+1}: {avg_loss:.4f}")

        # --- LIVE SAVE PER EPOCH ---
        # 1. Load current file content (if exists)
        current_data = {}
        if os.path.exists(json_path):
            try:
                with open(json_path, 'r') as f:
                    current_data = json.load(f)
            except: pass # Handle rare case of empty/corrupt file

        # 2. Update the specific dataset key
        current_data[dataset_name] = history

        # 3. Write back to Drive immediately
        with open(json_path, 'w') as f:
            json.dump(current_data, f, indent=4)
    return history

In [ ]:
# 4. Running and Final Plotting
datasets = ['FashionMNIST', 'CIFAR10', 'SVHN']
opts = ['SGD', 'Adam', 'Ada-STORM']

In [ ]:
non_ada_storm_results = {ds: {opt: train_model(ds, opt) for opt in opts if opt != 'Ada-STORM' and (opt != 'SGD' or ds != 'FashionMNIST')} for ds in datasets}



FashionMNIST | Adam | Epoch 1: 0.4116
FashionMNIST | Adam | Epoch 2: 0.2570
FashionMNIST | Adam | Epoch 3: 0.2085
FashionMNIST | Adam | Epoch 4: 0.1764
FashionMNIST | Adam | Epoch 5: 0.1466
FashionMNIST | Adam | Epoch 6: 0.1191
FashionMNIST | Adam | Epoch 7: 0.0953
FashionMNIST | Adam | Epoch 8: 0.0771
FashionMNIST | Adam | Epoch 9: 0.0586
FashionMNIST | Adam | Epoch 10: 0.0516
FashionMNIST | Adam | Epoch 11: 0.0405
FashionMNIST | Adam | Epoch 12: 0.0371
FashionMNIST | Adam | Epoch 13: 0.0336
FashionMNIST | Adam | Epoch 14: 0.0259
FashionMNIST | Adam | Epoch 15: 0.0275
FashionMNIST | Adam | Epoch 16: 0.0288
FashionMNIST | Adam | Epoch 17: 0.0220
FashionMNIST | Adam | Epoch 18: 0.0238
FashionMNIST | Adam | Epoch 19: 0.0218
FashionMNIST | Adam | Epoch 20: 0.0197
FashionMNIST | Adam | Epoch 21: 0.0162
FashionMNIST | Adam | Epoch 22: 0.0188
FashionMNIST | Adam | Epoch 23: 0.0199
FashionMNIST | Adam | Epoch 24: 0.0168
FashionMNIST | Adam | Epoch 25: 0.0153
FashionMNIST | Adam | Epoch 26: 0.

In [ ]:
ada_storm_results = {ds: train_model(ds, 'Ada-STORM') for ds in datasets}


100%|██████████| 26.4M/26.4M [00:02<00:00, 12.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 190kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.48MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.0MB/s]


FashionMNIST | Ada-STORM | Epoch 1: 0.5550
FashionMNIST | Ada-STORM | Epoch 2: 0.3120
FashionMNIST | Ada-STORM | Epoch 3: 0.2616
FashionMNIST | Ada-STORM | Epoch 4: 0.2306
FashionMNIST | Ada-STORM | Epoch 5: 0.2074
FashionMNIST | Ada-STORM | Epoch 6: 0.1865
FashionMNIST | Ada-STORM | Epoch 7: 0.1687
FashionMNIST | Ada-STORM | Epoch 8: 0.1522
FashionMNIST | Ada-STORM | Epoch 9: 0.1354
FashionMNIST | Ada-STORM | Epoch 10: 0.1206
FashionMNIST | Ada-STORM | Epoch 11: 0.1069
FashionMNIST | Ada-STORM | Epoch 12: 0.0933
FashionMNIST | Ada-STORM | Epoch 13: 0.0801
FashionMNIST | Ada-STORM | Epoch 14: 0.0679
FashionMNIST | Ada-STORM | Epoch 15: 0.0575


100%|██████████| 170M/170M [00:13<00:00, 12.6MB/s]


CIFAR10 | Ada-STORM | Epoch 1: 1.6525
CIFAR10 | Ada-STORM | Epoch 2: 1.1574
CIFAR10 | Ada-STORM | Epoch 3: 0.9041
CIFAR10 | Ada-STORM | Epoch 4: 0.7228
CIFAR10 | Ada-STORM | Epoch 5: 0.5731
CIFAR10 | Ada-STORM | Epoch 6: 0.4362
CIFAR10 | Ada-STORM | Epoch 7: 0.3006
CIFAR10 | Ada-STORM | Epoch 8: 0.1956
CIFAR10 | Ada-STORM | Epoch 9: 0.1152
CIFAR10 | Ada-STORM | Epoch 10: 0.0746
CIFAR10 | Ada-STORM | Epoch 11: 0.0468
CIFAR10 | Ada-STORM | Epoch 12: 0.0307
CIFAR10 | Ada-STORM | Epoch 13: 0.0265
CIFAR10 | Ada-STORM | Epoch 14: 0.0229
CIFAR10 | Ada-STORM | Epoch 15: 0.0132


100%|██████████| 182M/182M [01:26<00:00, 2.11MB/s]


SVHN | Ada-STORM | Epoch 1: 1.4754
SVHN | Ada-STORM | Epoch 2: 0.4129
SVHN | Ada-STORM | Epoch 3: 0.2963
SVHN | Ada-STORM | Epoch 4: 0.2333
SVHN | Ada-STORM | Epoch 5: 0.1826
SVHN | Ada-STORM | Epoch 6: 0.1390
SVHN | Ada-STORM | Epoch 7: 0.1003
SVHN | Ada-STORM | Epoch 8: 0.0713
SVHN | Ada-STORM | Epoch 9: 0.0487
SVHN | Ada-STORM | Epoch 10: 0.0320
SVHN | Ada-STORM | Epoch 11: 0.0312
SVHN | Ada-STORM | Epoch 12: 0.0303
SVHN | Ada-STORM | Epoch 13: 0.0152
SVHN | Ada-STORM | Epoch 14: 0.0090
SVHN | Ada-STORM | Epoch 15: 0.0086


NameError: name 'non_ada_storm_results' is not defined

In [ ]:

# NeurIPS Style Multi-Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plt.rcParams["font.family"] = "serif"

for i, ds in enumerate(datasets):
    for opt in opts:
        axes[i].plot(results[ds][opt], label=opt, marker='s', markersize=3)
    axes[i].set_title(ds)
    axes[i].set_xlabel('Epoch')
    axes[i].set_ylabel('Loss')
    axes[i].legend()

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
print(ada_storm_results)

NameError: name 'ada_storm_results' is not defined

In [ ]:

import json

file_path = "ada_storm_results.json"

with open(file_path, 'w') as json_file:
    json.dump(ada_storm_results, json_file, indent=4)

print(f"'ada_storm_results' dictionary successfully written to {file_path}")


'ada_storm_results' dictionary successfully written to ada_storm_results.json
